In [ ]:
import os
import os

folder_path = r"C:\project\political_ner\Final_NER"

try:
    # List all files in the folder
    files = os.listdir(folder_path)

    print("Files in the folder:")
    for file in files:
        print(file)
except FileNotFoundError:
    print(f"The folder at {folder_path} does not exist.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
import pandas as pd
import os

def create_country_glossary(input_file, output_folder):
    """
    1. Reads a single Excel file (input_file).
    2. Creates 'all_variations' by grouping over 'Names' or 'Party' and collecting unique 'NER'.
    3. Saves only the 'Glossary' sheet with duplicates removed and all required columns included.
    """

    # Read the input file
    df = pd.read_excel(input_file)

    # Ensure required columns exist; create them if missing
    for col in ['NER', 'Names', 'Party', 'NER_country_cleaned', 'EU']:
        if col not in df.columns:
            df[col] = ''

    # Fill NaNs for consistency
    df['NER'] = df['NER'].fillna('')
    df['Names'] = df['Names'].fillna('')
    df['Party'] = df['Party'].fillna('')
    df['NER_country_cleaned'] = df['NER_country_cleaned'].fillna('Unknown')
    df['EU'] = df['EU'].fillna('')

    # Get the country name from the first row (or 'Unknown' if empty)
    country_name = df['NER_country_cleaned'].iloc[0] if not df['NER_country_cleaned'].isnull().all() else 'Unknown'

    # Build separate glossaries
    name_glossary = (df[df['Names'] != '']
                     .groupby('Names')['NER']
                     .apply(lambda x: list(set(x)))
                     .to_dict())

    party_glossary = (df[(df['Names'] == '') & (df['Party'] != '')]
                      .groupby('Party')['NER']
                      .apply(lambda x: list(set(x)))
                      .to_dict())

    # Create a separate glossary DataFrame
    glossary_data = []
    for _, row in df.iterrows():
        glossary_data.append({
            'Type': 'Name' if row['Names'] else 'Party',
            'Entry': row['Names'] if row['Names'] else row['Party'],
            'Variations': ', '.join(name_glossary.get(row['Names'], []) if row['Names'] else party_glossary.get(row['Party'], [])),
            'Party': row['Party'],
            'Country': row['NER_country_cleaned'],
            'EU': row['EU']
        })

    # Convert to DataFrame and remove duplicates
    glossary_df = pd.DataFrame(glossary_data)
    glossary_df = glossary_df.drop_duplicates()  # Remove exact duplicates

    # Determine output file name: "<Country>_Glossary.xlsx"
    output_filename = f"{country_name}_Glossary.xlsx"
    output_path = os.path.join(output_folder, output_filename)

    # Save results to Excel with only one sheet: "Glossary"
    with pd.ExcelWriter(output_path) as writer:
        glossary_df.to_excel(writer, index=False, sheet_name='Glossary')

    print(f"Glossary saved to: {output_path}")


# Example usage:
if __name__ == "__main__":
    input_file_path = r"C:\project\political_ner\FINAL_NER_2025_17_01\NER_Identified\Final_SE_NER_Identified.xlsx"
    output_folder_path = (r"C:\project\political_ner"
                          r"\FINAL_NER_2025_17_01\Glossary")
    
    create_country_glossary(input_file_path, output_folder_path)
